# Lab 4: The same dream on Claude Managed Agents

**Cost: roughly $3 to $6 for the twelve-close record (Managed Agents adds session running time and a Sonnet lead
agent). Requires Managed
Agents beta access. Time: 15 minutes.**

Labs 2 and 3 ran the loop on your machine so every mechanic was visible. In production, Anthropic's cookbook runs it on
**Claude Managed Agents**:

| | Local mode (Labs 2-3) | Managed Agents mode (this lab) |
|---|---|---|
| Who runs the agent loop | `close_dreamer/agent.py` on your machine | Anthropic's orchestration layer |
| Orchestration | Python runs the phases | A **lead agent** (Sonnet) writes and starts a **dynamic workflow run** |
| Workers | analyzer + organizer via Messages API | the same, as **predefined agents** in a sandbox with no network |
| Memory | a local folder | a **memory store**: versioned, auditable, mounted read-write for dreaming and **read-only** for the close assistant |
| Prompts | adapted for the close and its source kinds | the cookbook's own, unchanged, plus run instructions for the close |

`close_dreamer/managed.py` follows the cookbook notebook cell for cell, using its helper module vendored unchanged.

Not the same as **Managed Agents Dreams** (research preview): that API takes a store plus up to 100 sessions and
writes a **new** store, leaving the input untouched. The cookbook's workflow, used here, updates a store in place and
lets you choose the prompts and models.

In [ ]:
import yaml

from close_dreamer import config, managed, memory
from close_dreamer import transcripts as tx

first = tx.select(tx.load(config.TRANSCRIPTS), through="2026-09")
truth = yaml.safe_load((config.DATA / "ground_truth.yaml").read_text(encoding="utf-8"))
probes = dict(extra_secrets=[truth["secret"]["text"]],
              extra_injections=[i["text"] for i in truth["injected_instructions"]])
print(managed.CLOSE_INSTRUCTIONS)

In [ ]:
# Paid cell: creates a memory store, agents, an environment and a session; cleans up all but the store.
out_dir = managed.run(first, budget_usd=5.0)

In [ ]:
print((out_dir / "_index.md").read_text(encoding="utf-8") if (out_dir / "_index.md").exists() else "no index")
print(memory.lint(out_dir, source_bytes=sum(t.size for t in first), **probes).summary())

## Dream again after October, updating the same store

Copy the store id printed above (`memstore_...`) into `STORE_ID`. The run updates the store in place; every write is a
memory version you can audit in the Console for 30 days.

In [ ]:
STORE_ID = ""  # e.g. "memstore_..."
if STORE_ID:
    second = tx.select(tx.load(config.TRANSCRIPTS), through="2026-10", after="2026-09")
    out_dir2 = managed.run(second, budget_usd=5.0, store_id=STORE_ID)
    print(memory.diff(out_dir, out_dir2)[:6000])

## Compare with local mode

Evaluate the downloaded store exactly like the local playbook:

```
close-dreamer evaluate --conditions playbook --playbook output/managed/<store name>
```

**Reflect:** which parts of the managed setup would you want in an enterprise deployment (versioned memory, read-only
mounts, a session budget, no network), and which parts of local mode were more useful for learning?